# Fase 1: Preprocesado del Dataset
## Proyecto: Clasificador Over/Under 2.5 goles en partidos de fútbol

En este notebook se prepara el dataset para entrenar un modelo de clasificación binaria que predice si un partido de fútbol tendrá más de 2.5 goles (Over) o no (Under).

Fuentes de apoyo: Titanic: A Complete Beginner's Guide (https://www.kaggle.com/code/reighns/titanic-a-complete-beginner-s-guide).

## 1. Importar librerías

In [1]:
# Librerías para manejo de datos
import pandas as pd
import numpy as np

# Librerías para visualización
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías de sklearn para el preprocesado
from sklearn.model_selection import train_test_split  # para separar train/test
from sklearn.preprocessing import StandardScaler      # para escalar las features

# Configuración de los gráficos
plt.rcParams['figure.figsize'] = (10, 5)
sns.set(style='whitegrid')

## 2. Carga y exploración inicial

El dataset se obtuvo del repositorio público **Kaggle: Club Football Match Data (2000-2025)**, que contiene partidos de 38 ligas de fútbol con estadísticas, rating ELO y cuotas de bookmakers.

Como el dataset ya viene completo y con muchas instancias, **no es necesario aumentar el set de datos**.

In [2]:
# Cargar el CSV
df = pd.read_csv('Matches.csv', low_memory=False)

print('Filas:', len(df))
print('Columnas:', df.shape[1])

df.head()

Filas: 230557
Columnas: 48


,Division,MatchDate,MatchTime,HomeTeam,AwayTeam,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,...,MaxUnder25,HandiSize,HandiHome,HandiAway,C_LTH,C_LTA,C_VHD,C_VAD,C_HTB,C_PHB
0,F1,2000-07-28,NaN,Marseille,Troyes,1686.34,1586.57,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,F1,2000-07-28,NaN,Paris SG,Strasbourg,1714.89,1642.51,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,F2,2000-07-28,NaN,Wasquehal,Nancy,1465.08,1633.80,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,F1,2000-07-29,NaN,Auxerre,Sedan,1635.58,1624.22,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,F1,2000-07-29,NaN,Bordeaux,Metz,1734.34,1673.11,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Selección de columnas de interés

 Seleccionamos las columnas que vamos a usar:
 - Division: la liga donde se juega el partido (categórica)
 - HomeElo, AwayElo: rating ELO de cada equipo (numéricas, mide la fuerza del equipo)
 - Form3Home, Form5Home, Form3Away, Form5Away: puntos en los últimos 3 y 5 partidos
 - FTHome, FTAway: goles finales del partido que se usaran para crear el target

In [3]:
columnas_interes = [
    'Division', 'HomeElo', 'AwayElo', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'FTHome', 'FTAway'
]

df = df[columnas_interes]
print('Columnas seleccionadas:', {df.shape[1]})
df.head()

Columnas seleccionadas: {9}


,Division,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,FTHome,FTAway
0,F1,1686.34,1586.57,0.0,0.0,0.0,0.0,3.0,1.0
1,F1,1714.89,1642.51,0.0,0.0,0.0,0.0,3.0,1.0
2,F2,1465.08,1633.80,0.0,0.0,0.0,0.0,0.0,1.0
3,F1,1635.58,1624.22,0.0,0.0,0.0,0.0,0.0,1.0
4,F1,1734.34,1673.11,0.0,0.0,0.0,0.0,1.0,1.0


## 4. Limpieza de los datos: manejo de nulos por columna

El tutorial Titanic explica que no es buena idea eliminar todos los registros con nulos a ciegas, ni rellenarlos todos con la mediana sin pensar. Depende de cada columna.

In [4]:
# Revisar nulos de las columnas seleccionadas
df[columnas_interes].isnull().sum()

,0
Division,0
HomeElo,88960
AwayElo,89029
Form3Home,1500
Form5Home,1500
Form3Away,1500
Form5Away,1500
FTHome,3
FTAway,3


**Decisiones:**
- 'HomeElo' y 'AwayElo' tienen ~38% de nulos (demasiados). Eliminamos esas filas porque imputar tantos valores distorsionaría la distribución.
- 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away' tienen menos del 1% de nulos. Los imputamos con la mediana.
- 'FTHome' y 'FTAway' los necesitamos para crear el target, así que eliminamos las filas que los tengan nulos (son muy pocas).

In [5]:
# 1. Eliminar filas sin ELO (muchos nulos, imputar distorsionaría)
filas_antes = len(df)
df = df.dropna(subset=['HomeElo', 'AwayElo'])
print(f'Filas eliminadas por ELO nulo: {filas_antes - len(df):,}')
print(f'Filas restantes: {len(df):,}')

Filas eliminadas por ELO nulo: 98,144
Filas restantes: 132,413


In [6]:
# 2. Imputar nulos en las columnas Form con la mediana
form_cols = ['Form3Home', 'Form5Home', 'Form3Away', 'Form5Away']

for col in form_cols:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f'{col}: imputado con mediana = {mediana}')

print(f'\nNulos restantes en columnas Form: {df[form_cols].isnull().sum().sum()}')

Form3Home: imputado con mediana = 4.0
Form5Home: imputado con mediana = 7.0
Form3Away: imputado con mediana = 4.0
Form5Away: imputado con mediana = 7.0

Nulos restantes en columnas Form: 0


In [7]:
# 3. Eliminar las pocas filas sin goles registrados (necesarias para el target)
df = df.dropna(subset=['FTHome', 'FTAway'])
print(f'Filas finales: {len(df):,}')

Filas finales: 132,411


## 5. Creación de la variable objetivo (target)

Como queremos predecir si el partido tendrá más de 2.5 goles, primero sumamos los goles de ambos equipos. Luego creamos una columna binaria:
- **1** si hubo más de 2.5 goles (Over)
- **0** si hubo 2.5 goles o menos (Under)

In [8]:
# Sumar los goles del partido
df['TotalGoles'] = df['FTHome'] + df['FTAway']

# Crear el target con 1 si goles > 2.5, 0 si no
df['Over25'] = (df['TotalGoles'] > 2.5).astype(int)

# revisar si esta balanceado
print('Distribución del target:')
print(df['Over25'].value_counts())
print('\nEn porcentaje:')
print(df['Over25'].value_counts(normalize=True).round(3)*100)

Distribución del target:
Over25
0    67617
1    64794
Name: count, dtype: int64

En porcentaje:
Over25
0    51.1
1    48.9
Name: proportion, dtype: float64


El target está prácticamente balanceado (~49% Over, ~51% Under), lo cual es bueno porque significa que el modelo no tendrá un sesgo natural hacia una clase. No necesitamos aplicar técnicas de balanceo.

In [9]:
##Eliminamos las columnas que solo usamos para crearlo (Esto no se puede correr dos veces seguidas sin volver a correr el codigo arriba)
# (FTHome, FTAway y TotalGoles ya no las necesitamos)
df = df.drop(columns=['FTHome', 'FTAway', 'TotalGoles'])
print('Columnas actuales:')
print(df.columns.tolist())

Columnas actuales:
['Division', 'HomeElo', 'AwayElo', 'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away', 'Over25']


## 6. Feature engineering: crear EloDiff

Agregamos una feature derivada: la diferencia entre el ELO del equipo local y el del visitante. Esto le da al modelo una pista directa de qué tan desbalanceado es el partido (si la diferencia es grande, probablemente el favorito meta muchos goles; si es cercana a 0, los equipos están parejos).

In [10]:
# Diferencia de ELO: positivo significa que el local es más fuerte
df['EloDiff'] = df['HomeElo'] - df['AwayElo']

df[['HomeElo', 'AwayElo', 'EloDiff']].round(2)

,HomeElo,AwayElo,EloDiff
0,1686.34,1586.57,99.77
1,1714.89,1642.51,72.38
2,1465.08,1633.80,-168.72
3,1635.58,1624.22,11.36
4,1734.34,1673.11,61.23
...,...,...,...
230552,1339.21,1544.15,-204.94
230553,1544.16,1433.67,110.49
230554,1473.67,1569.98,-96.31
230555,1574.90,1525.76,49.14


## 7. Preprocesado de la variable categórica: One-Hot Encoding

La columna **Division** tiene los códigos de las ligas (por ejemplo, E0 = Premier League, SP1 = La Liga, etc.). Como los modelos de machine learning trabajan con números, tenemos que convertir esta variable categórica en variables binarias.

El tutorial Titanic sugiere el siguiente metodo para nuestro contexto:
- **One-Hot Encoding**: crea una columna binaria por cada categoría, sin asumir un orden. Como las ligas no tienen un orden jerárquico entre ellas, este metodo nos sirve.

In [11]:
# Ver cuántas ligas hay
print(f'Cantidad de ligas: {df['Division'].nunique()}')
print(f'Ejemplos: {df['Division'].unique()[10:]}')

Cantidad de ligas: 26
Ejemplos: ['E2' 'I2' 'SP2' 'SP1' 'I1' 'SC0' 'G1' 'SC1' 'NOR' 'DEN' 'AUT' 'SWE' 'POL'
 'ROM' 'FIN' 'RUS']


In [12]:
# Aplicamos One-Hot Encoding a la columna Division
# pd.get_dummies crea una columna binaria por cada categoría
# prefix='Liga' agrega 'Liga_' al inicio del nombre de cada columna nueva (ej: Liga_E0, Liga_SP1)
# dtype=int hace que las columnas sean 0/1 directamente (en vez de True/False)
df = pd.get_dummies(df, columns=['Division'], prefix='Liga', dtype=int)

# Contar cuántas columnas de liga se crearon
columnas_liga = [c for c in df.columns if c.startswith('Liga_')]

print(f'Total de columnas después de One-Hot: {df.shape[1]}')
print(f'Columnas de liga creadas: {len(columnas_liga)}')
df.head()

Total de columnas después de One-Hot: 34
Columnas de liga creadas: 26


,HomeElo,AwayElo,Form3Home,Form5Home,Form3Away,Form5Away,Over25,EloDiff,Liga_AUT,Liga_B1,...,Liga_P1,Liga_POL,Liga_ROM,Liga_RUS,Liga_SC0,Liga_SC1,Liga_SP1,Liga_SP2,Liga_SWE,Liga_T1
0,1686.34,1586.57,0.0,0.0,0.0,0.0,1,99.77,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1714.89,1642.51,0.0,0.0,0.0,0.0,1,72.38,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1465.08,1633.80,0.0,0.0,0.0,0.0,0,-168.72,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1635.58,1624.22,0.0,0.0,0.0,0.0,0,11.36,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1734.34,1673.11,0.0,0.0,0.0,0.0,0,61.23,0,0,...,0,0,0,0,0,0,0,0,0,0


## 8. Separación en features (X) y target (y)

Antes de hacer el split train/test, separamos las features (X) del target (y).

In [13]:
# y = lo que queremos predecir (el target)
y = df['Over25']

# X = todas las demás columnas (las features)
X = df.drop(columns=['Over25'])

print(f'Shape de X (features): {X.shape}')
print(f'Shape de y (target):   {y.shape}')

Shape de X (features): (132411, 33)
Shape de y (target):   (132411,)


## 9. Separación en sets de entrenamiento y prueba

Usamos `train_test_split` de sklearn para dividir los datos:
- **80%** para entrenar el modelo (`x_train`)
- **20%** para evaluarlo (`x_test`)

Usamos `random_state=0` para que la división sea reproducible (que dé el mismo resultado cada vez que corramos el notebook), igual que en el tutorial Titanic.

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    train_size=0.8,
    test_size=0.2,
    random_state=0
)

print(f'x_train: {x_train.shape}, y_train: {y_train.shape}')
print(f'x_test:  {x_test.shape}, y_test:  {y_test.shape}')
print()
print(f'Proporción de Over en train: {y_train.mean():.3f}')
print(f'Proporción de Over en test:  {y_test.mean():.3f}')

x_train: (105928, 33), y_train: (105928,)
x_test:  (26483, 33), y_test:  (26483,)

Proporción de Over en train: 0.489
Proporción de Over en test:  0.489


## 10. Escalamiento de las features numéricas

Las features numéricas (ELO, Forma) están en escalas muy distintas:
- El ELO va de ~1200 a ~2100
- La forma va de 0 a 9 o de 0 a 15

Si dejamos los datos así, las features con valores más grandes (como ELO) tendrán más peso artificialmente.

Las columnas de liga (Liga_E0, Liga_SP1, etc.) NO se escalan porque ya son 0/1.

In [15]:
# Lista de columnas numéricas que sí queremos escalar
columnas_numericas = [
    'HomeElo', 'AwayElo', 'EloDiff',
    'Form3Home', 'Form5Home', 'Form3Away', 'Form5Away'
]

# Crear el scaler
sc = StandardScaler()

# Hacemos copias para no modificar los originales
x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()

# Ajustar y transformar las columnas numéricas en train
x_train_scaled[columnas_numericas] = sc.fit_transform(x_train[columnas_numericas])

# Aplicar la misma transformación al test (sin volver a ajustar)
x_test_scaled[columnas_numericas] = sc.transform(x_test[columnas_numericas])

print('Estadísticas de las features numéricas despues d escalar (train):')
print(x_train_scaled[columnas_numericas].describe().round(3).loc[['mean', 'std', 'min', 'max']])

Estadísticas de las features numéricas despues d escalar (train):
      HomeElo  AwayElo  EloDiff  Form3Home  Form5Home  Form3Away  Form5Away
mean    0.000   -0.000    0.000      0.000     -0.000     -0.000     -0.000
std     1.000    1.000    1.000      1.000      1.000      1.000      1.000
min    -2.830   -2.829   -4.227     -1.668     -2.053     -1.778     -2.131
max     3.728    3.748    4.162      2.127      2.542      1.996      2.457


In [ ]:
#Este bloque de codigo es para guardar los csv
import os
os.makedirs('data_procesada', exist_ok=True)
x_train_scaled.to_csv('data_procesada/x_train.csv', index=False)
x_test_scaled.to_csv('data_procesada/x_test.csv', index=False)
y_train.to_csv('data_procesada/y_train.csv', index=False)
y_test.to_csv('data_procesada/y_test.csv', index=False)